# Imports

In [ ]:
import os
import json
import time

from tqdm import tqdm

from dotenv import load_dotenv

from openai import OpenAI

from langchain_core.documents import Document

from langchain_chroma import Chroma

from langchain_community.embeddings import HuggingFaceEmbeddings

# Configuration

In [151]:
load_dotenv()
NVIDIA_API_KEY=os.getenv("NVIDIA_API_KEY")

In [152]:

persist_directory = "./tesla_db"

chunk_collection_name = "tesla-10k-2019-to-2023"

hq_collection_name = "tesla-hypothetical-questions"

embedding_model_name = (
    "sentence-transformers/all-mpnet-base-v2"
)

QUESTIONS_PER_CHUNK = 3

BATCH_SIZE = 10


HQ_GENERATION_MODEL = "meta/llama-3.1-8b-instruct"

ANSWER_MODEL = "meta/llama-3.1-8b-instruct"

# Initialize Groq Client

In [153]:
# Make sure NVIDIA_API_KEY is available

client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=os.getenv("NVIDIA_API_KEY")
)

print("NVIDIA NIM Client Initialized")

NVIDIA NIM Client Initialized


# Initialize Embedding Model

In [154]:
embeddings = HuggingFaceEmbeddings(
    model_name=embedding_model_name
)

print("Embedding Model Loaded")

Embedding Model Loaded


# Connect to Existing Chunk Collection

In [155]:
chunk_vectorstore = Chroma(
    collection_name=chunk_collection_name,
    persist_directory=persist_directory,
    embedding_function=embeddings
)

print("Connected to Chunk Collection")

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Connected to Chunk Collection


# Connect to HQ Collection

This collection will store generated hypothetical questions.

In [156]:
hq_vectorstore = Chroma(
    collection_name=hq_collection_name,
    persist_directory=persist_directory,
    embedding_function=embeddings
)

print("Connected to HQ Collection")

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Connected to HQ Collection


# Load Existing Chunks

In [157]:
all_chunk_data = chunk_vectorstore.get(
    include=["documents", "metadatas"]
)

print(
    f"Total Chunks Found: "
    f"{len(all_chunk_data['ids'])}"
)

Total Chunks Found: 3337


# Verify Chunk IDs

In [158]:
for idx in range(5):

    print(
        all_chunk_data["ids"][idx]
    )

text_0
text_1
text_2
text_3
text_4


# Create Chunk Dictionary

- Format:

- {
    - "text_0" :  "...",
    - "text_1" :  "...",
- }

In [159]:
chunk_dict = {}

for chunk_id, chunk_text in zip(
    all_chunk_data["ids"],
    all_chunk_data["documents"]
):

    chunk_dict[chunk_id] = chunk_text

print(
    f"Chunk Dictionary Created: "
    f"{len(chunk_dict)} chunks"
)

Chunk Dictionary Created: 3337 chunks


# Utility Function: Batch Generator

This function creates batches of 10 chunks
to reduce API calls and avoid rate limits.

In [160]:
def create_batches(data_dict, batch_size=10):

    items = list(data_dict.items())

    for i in range(
        0,
        len(items),
        batch_size
    ):

        yield dict(
            items[i:i + batch_size]
        )

# Verify Batching Logic

In [161]:
sample_batch = next(
    create_batches(
        chunk_dict,
        batch_size=BATCH_SIZE
    )
)

print(
    f"Chunks In Batch: "
    f"{len(sample_batch)}"
)

print(
    list(sample_batch.keys())
)

Chunks In Batch: 10
['text_0', 'text_1', 'text_2', 'text_3', 'text_4', 'text_5', 'text_6', 'text_7', 'text_8', 'text_9']


# Prompt Template for Hypothetical Question Generation

The model receives 10 chunks and generates
exactly 3 questions per chunk.

Output must be valid JSON only.

In [162]:
HQ_GENERATION_SYSTEM_PROMPT = """
You are an expert at generating retrieval-oriented questions.

You will receive multiple document chunks.

For EACH chunk:

1. Generate exactly 3 hypothetical questions.
2. Questions must be answerable from the chunk.
3. Questions should resemble realistic user queries.
4. Questions should improve retrieval quality.
5. Return ONLY valid JSON.
6. Do not include markdown.
7. Do not include explanations.

Output format:

{
  "text_0": [
    "question 1",
    "question 2",
    "question 3"
  ],
  "text_1": [
    "question 1",
    "question 2",
    "question 3"
  ]
}
"""

# Utility Function: Build Batch Prompt

In [163]:
def build_batch_prompt(chunk_batch):

    prompt = []

    prompt.append(
        "Generate exactly 3 questions for each chunk.\n"
    )

    for chunk_id, chunk_text in chunk_batch.items():

        prompt.append(
            f"\nChunk ID: {chunk_id}\n"
        )

        prompt.append(
            f"Chunk Text:\n{chunk_text}\n"
        )

    return "\n".join(prompt)

# Utility Function: Groq API Call

Includes retry logic for:
- Rate Limits
- Temporary Failures

In [164]:
def generate_hq_batch(
    chunk_batch,
    max_retries=5,
    retry_delay=10
):

    prompt = build_batch_prompt(
        chunk_batch
    )

    for attempt in range(max_retries):

        try:

            response = client.chat.completions.create(
                model=HQ_GENERATION_MODEL,
                messages=[
                    {
                        "role": "system",
                        "content": HQ_GENERATION_SYSTEM_PROMPT
                    },
                    {
                        "role": "user",
                        "content": prompt
                    }
                ],
                temperature=0
            )

            return (
                response
                .choices[0]
                .message
                .content
            )

        except Exception as e:

            print(
                f"Attempt {attempt+1} failed:"
            )

            print(str(e))

            if attempt < max_retries - 1:

                print(
                    f"Retrying in "
                    f"{retry_delay} seconds..."
                )

                time.sleep(
                    retry_delay
                )

            else:

                raise

# Utility Function: Parse JSON Response

In [165]:
def parse_hq_json(
    response_text
):

    try:

        return json.loads(
            response_text
        )

    except Exception:

        cleaned_response = (
            response_text
            .replace("```json", "")
            .replace("```", "")
            .strip()
        )

        return json.loads(
            cleaned_response
        )

# Test HQ Generation on First Batch

In [166]:
first_batch = next(
    create_batches(
        chunk_dict,
        batch_size=BATCH_SIZE
    )
)

test_response = generate_hq_batch(
    first_batch
)

print(test_response[:200])

{
  "text_0": [
    "What is the name of the company that filed this report?",
    "What is the fiscal year ended for this report?",
    "What is the purpose of this amendment to the original Form 10-


# Verify JSON Parsing

In [167]:
parsed_response = parse_hq_json(
    test_response
)

print(
    parsed_response.keys()
)

dict_keys(['text_0', 'text_1', 'text_2', 'text_3', 'text_4', 'text_5', 'text_6', 'text_7', 'text_8', 'text_9'])


# Generate HQs For All Chunks

In [168]:
all_generated_questions = {}

batch_counter = 0

total_batches = (
    len(chunk_dict)
    + BATCH_SIZE - 1
) // BATCH_SIZE

for chunk_batch in tqdm(
    create_batches(
        chunk_dict,
        batch_size=BATCH_SIZE
    ),
    total=total_batches
):

    batch_counter += 1

    try:

        response_text = (
            generate_hq_batch(
                chunk_batch
            )
        )

        parsed_json = (
            parse_hq_json(
                response_text
            )
        )

        all_generated_questions.update(
            parsed_json
        )

    except Exception as e:

        print(
            f"Batch {batch_counter}"
            f" failed."
        )

        print(str(e))

  8%|▊         | 26/334 [03:08<33:59,  6.62s/it]

Batch 26 failed.
Expecting ',' delimiter: line 51 column 4 (char 6678)


 50%|█████     | 167/334 [1:07:51<3:23:07, 72.98s/it] 

Batch 167 failed.
Expecting ',' delimiter: line 51 column 4 (char 3873)


 51%|█████     | 170/334 [1:08:12<1:20:37, 29.50s/it]

...

 82%|████████▏ | 274/334 [1:58:40<21:34, 21.57s/it]   

Attempt 1 failed:
Request timed out.
Retrying in 10 seconds...


 93%|█████████▎| 309/334 [2:34:49<03:51,  9.27s/it]   

Batch 309 failed.
Expecting ',' delimiter: line 35 column 112 (char 2810)


100%|██████████| 334/334 [2:38:56<00:00, 28.55s/it]


# Verify Question Generation

In [169]:
print(
    f"Chunks Processed: "
    f"{len(all_generated_questions)}"
)

first_key = next(
    iter(all_generated_questions)
)

print(
    first_key
)

print(
    all_generated_questions[
        first_key
    ]
)

Chunks Processed: 3268
text_0
['What is the name of the company that filed this report?', 'What is the fiscal year ended for this report?', 'What is the purpose of this amendment to the original Form 10-K?']


# Convert Questions to LangChain Documents

ID Format:

- hq_text_0_0
- hq_text_0_1
- hq_text_0_2

Metadata:

- {
    - "parent_chunk_id": "text_0"
- }

In [170]:
hq_documents = []

hq_ids = []

for chunk_id, questions in (
    all_generated_questions.items()
):

    for idx, question in enumerate(
        questions
    ):

        hq_id = (
            f"hq_{chunk_id}_{idx}"
        )

        hq_doc = Document(
            page_content=question,
            metadata={
                "hq_id": hq_id,
                "parent_chunk_id": chunk_id
            }
        )

        hq_documents.append(
            hq_doc
        )

        hq_ids.append(
            hq_id
        )

# Verify HQ Documents

In [171]:
print(
    f"Total HQ Documents: "
    f"{len(hq_documents)}"
)

hq_documents[:3]

Total HQ Documents: 9802


[Document(metadata={'hq_id': 'hq_text_0_0', 'parent_chunk_id': 'text_0'}, page_content='What is the name of the company that filed this report?'),
 Document(metadata={'hq_id': 'hq_text_0_1', 'parent_chunk_id': 'text_0'}, page_content='What is the fiscal year ended for this report?'),
 Document(metadata={'hq_id': 'hq_text_0_2', 'parent_chunk_id': 'text_0'}, page_content='What is the purpose of this amendment to the original Form 10-K?')]

# Store HQ Documents in Chroma

### check for 

In [174]:
print(
    "Total IDs:",
    len(hq_ids)
)

print(
    "Unique IDs:",
    len(set(hq_ids))
)

Total IDs: 9802
Unique IDs: 9802


In [175]:
INSERT_BATCH_SIZE = 1000

for i in range(
    0,
    len(hq_documents),
    INSERT_BATCH_SIZE
):

    batch_docs = hq_documents[
        i:i + INSERT_BATCH_SIZE
    ]

    batch_ids = hq_ids[
        i:i + INSERT_BATCH_SIZE
    ]

    hq_vectorstore.add_documents(
        documents=batch_docs,
        ids=batch_ids
    )

    print(
        f"Inserted "
        f"{len(batch_docs)} docs "
        f"({i} -> "
        f"{min(i + INSERT_BATCH_SIZE, len(hq_documents))})"
    )

Inserted 1000 docs (0 -> 1000)
Inserted 1000 docs (1000 -> 2000)
Inserted 1000 docs (2000 -> 3000)
Inserted 1000 docs (3000 -> 4000)
Inserted 1000 docs (4000 -> 5000)
Inserted 1000 docs (5000 -> 6000)
Inserted 1000 docs (6000 -> 7000)
Inserted 1000 docs (7000 -> 8000)
Inserted 1000 docs (8000 -> 9000)
Inserted 802 docs (9000 -> 9802)


# Verify HQ Collection

In [176]:
hq_data = hq_vectorstore.get()

print(
    f"Stored HQ Records: "
    f"{len(hq_data['ids'])}"
)

Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argument but 3 were given


Stored HQ Records: 9802


# Sample HQ Record

In [177]:
sample_index = 0

print(
    hq_data["ids"][sample_index]
)

print(
    hq_data["metadatas"][sample_index]
)

print(
    hq_data["documents"][sample_index]
)

hq_text_0_0
{'hq_id': 'hq_text_0_0', 'parent_chunk_id': 'text_0'}
What is the name of the company that filed this report?


# Create HQ Retriever

The retriever searches the hypothetical question collection.

The returned questions contain metadata with:

{
    "parent_chunk_id": "text_x"
}

In [178]:
hq_retriever = hq_vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 5
    }
)

print("HQ Retriever Created")

HQ Retriever Created


# User Query

In [179]:
user_query = (
    "What was Tesla automotive revenue in 2021?"
)

print(user_query)

What was Tesla automotive revenue in 2021?


# Retrieve Similar Hypothetical Questions

In [180]:
retrieved_hq_documents = (
    hq_retriever.invoke(
        user_query
    )
)

print(
    f"Retrieved HQ Documents: "
    f"{len(retrieved_hq_documents)}"
)

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


Retrieved HQ Documents: 5


# Inspect Retrieved Questions

In [181]:
for idx, doc in enumerate(
    retrieved_hq_documents,
    start=1
):

    print(
        f"\n----- HQ {idx} -----"
    )

    print(
        f"HQ ID: "
        f"{doc.metadata['hq_id']}"
    )

    print(
        f"Parent Chunk: "
        f"{doc.metadata['parent_chunk_id']}"
    )

    print(
        f"Question: "
        f"{doc.page_content}"
    )


----- HQ 1 -----
HQ ID: hq_text_30_0
Parent Chunk: text_30
Question: What were Tesla's total revenues in 2021?

----- HQ 2 -----
HQ ID: hq_text_2390_1
Parent Chunk: text_2390
Question: How much did Tesla's automotive sales revenue increase from 2020 to 2022?

----- HQ 3 -----
HQ ID: hq_text_2390_0
Parent Chunk: text_2390
Question: What were Tesla's revenues in 2022?

----- HQ 4 -----
HQ ID: hq_text_1061_0
Parent Chunk: text_1061
Question: What are the revenues of Tesla, Inc. for the year 2020?

----- HQ 5 -----
HQ ID: hq_text_2393_2
Parent Chunk: text_2393
Question: What was the net income for Tesla in 2022?


# Extract Parent Chunk IDs

In [182]:
parent_chunk_ids = []

for doc in retrieved_hq_documents:

    parent_chunk_ids.append(
        doc.metadata[
            "parent_chunk_id"
        ]
    )

print(parent_chunk_ids)

['text_30', 'text_2390', 'text_2390', 'text_1061', 'text_2393']


# Remove Duplicate Parent IDs

In [183]:
unique_parent_ids = list(
    dict.fromkeys(
        parent_chunk_ids
    )
)

print(
    f"Unique Parent IDs: "
    f"{len(unique_parent_ids)}"
)

print(unique_parent_ids)

Unique Parent IDs: 4
['text_30', 'text_2390', 'text_1061', 'text_2393']


# Retrieve Original Chunks

In [184]:
original_chunks = (
    chunk_vectorstore.get(
        ids=unique_parent_ids,
        include=[
            "documents",
            "metadatas"
        ]
    )
)

# Verify Retrieved Chunks

In [185]:
print(
    f"Chunks Retrieved: "
    f"{len(original_chunks['ids'])}"
)

Chunks Retrieved: 4


In [ ]:
for idx in range(len(original_chunks["ids"])):

    print("\n==================")

    print(original_chunks["ids"][idx])

    print(original_chunks["documents"][idx][:500])

    print("\n==================")


text_30
objectives.	In	2021,	Tesla’s	full-year	accomplishments	under	our	executive	leadership	included	the	following:
	
•
Total	revenues	of	$53.82	billion,	representing	an	increase	of	$22.28	billion,	or	70.64%	compared	to	the	prior	year;
	
•
Net	income	attributable	to	common	stockholders	of	$5.52	billion	and	an	operating	margin	of	12.1%,	representing	favorable	changes	of
$4.80	billion	and	5.8%,	respectively,	compared	to	the	prior	year;
	
•
Annual	vehicle	delivery	and	production	records	of	936,222	and	
9


text_1061
Tesla,	Inc.
	
Consolidated	Statements	of	Operations
(in	millions,	except	per	share	data)
	
	
	
Year	Ended	December	31,
	
	
	
2020
	
	
2019
	
	
2018
	
Revenues
	
	
	
	
	
	
	
	
	
	
	
	
Automotive	sales
	
$
26,184
	
	
$
19,952
	
	
$
17,632
	
Automotive	leasing
	
	
1,052
	
	
	
869
	
	
	
883
	
Total	automotive	revenues
	
	
27,236
	
	
	
20,821
	
	
	
18,515
	
Energy	generation	and	storage
	
	
1,994
	
	
	
1,531
	
	
	
1,555
	
Services	and	other
	
	
2,306
	
	
	
2,226
	
	
	
1,391
	
To

# Build Context

The retrieved chunks are merged into a single
context string for final answer generation.

In [189]:
context = "\n\n---\n\n".join(
    original_chunks["documents"]
)

print(f"Context Length: {len(context)} characters")

Context Length: 3508 characters


# Preview Context

In [191]:
print(context[:2000])

objectives.	In	2021,	Tesla’s	full-year	accomplishments	under	our	executive	leadership	included	the	following:
	
•
Total	revenues	of	$53.82	billion,	representing	an	increase	of	$22.28	billion,	or	70.64%	compared	to	the	prior	year;
	
•
Net	income	attributable	to	common	stockholders	of	$5.52	billion	and	an	operating	margin	of	12.1%,	representing	favorable	changes	of
$4.80	billion	and	5.8%,	respectively,	compared	to	the	prior	year;
	
•
Annual	vehicle	delivery	and	production	records	of	936,222	and	
930,422
	total	vehicles,	representing	an	increase	of	87.38%	and	82.53%,
respectively,	compared	to	the	prior	year;
	
•
3.99	gigawatt	hours	of	energy	storage	and	345	megawatts	of	solar	energy	systems	deployed;	and
	
•
Ongoing	progress	in	the	global	growth	of	our	manufacturing	capabilities,	including	the	commencement	of	builds	of	the	Model	Y	in
Gigafactory	Texas	and	equipment	testing	through	the	vehicle	production	process	in	Gigafactory	Berlin.
7

---

Tesla,	Inc.
	
Consolidated	Statements	of	Operat

# Retrieval Pipeline Summary

In [192]:
print("\nUSER QUERY:")
print(user_query)

print("\nRETRIEVED HQ DOCUMENTS:")
print(len(retrieved_hq_documents))

print("\nUNIQUE PARENT CHUNKS:")
print(len(unique_parent_ids))

print("\nCONTEXT LENGTH:")
print(len(context))


USER QUERY:
What was Tesla automotive revenue in 2021?

RETRIEVED HQ DOCUMENTS:
5

UNIQUE PARENT CHUNKS:
4

CONTEXT LENGTH:
3508


# System Prompt

The model should answer only from the retrieved context.
If the answer is unavailable, it must return:
"I don't know"

In [193]:
QNA_SYSTEM_PROMPT = """
You are an assistant that answers questions about Tesla annual reports.

You will receive:

1. Context
2. User Question

Rules:

1. Use ONLY the provided context.
2. Do not use external knowledge.
3. If the answer is not found in the context,
   respond with:

   I don't know

4. Keep answers concise and factual.
5. Do not mention the context.
"""

# User Prompt Template

In [194]:
QNA_USER_TEMPLATE = """
<Context>

{context}

</Context>

<Question>

{question}

</Question>
"""

# Generate Final Answer

In [195]:
def generate_answer(context,question):

    final_prompt = [
        
        {
            "role": "system",
            "content": QNA_SYSTEM_PROMPT
        },

        {
            "role": "user",
            "content": QNA_USER_TEMPLATE.format(
                context=context,
                question=question
            )
        }
    ]

    response = client.chat.completions.create(
        model=ANSWER_MODEL,
        messages=final_prompt,
        temperature=0
    )

    return (response.choices[0].message.content)

# Test Final Answer Generation

In [196]:
answer = generate_answer(
    context=context,
    question=user_query
)

print(answer)

$47,232 million.


# End-to-End Hypothetical Question RAG

In [199]:
def answer_query(
    query,
    retrieval_k=5
):
    """
    End-to-End HQ RAG Pipeline
    """

    # ----------------------------------
    # Step 1
    # Retrieve HQ Documents
    # ----------------------------------

    retrieved_hq_docs = (hq_vectorstore.similarity_search(query=query,k=retrieval_k))

    # ----------------------------------
    # Step 2
    # Extract Parent IDs
    # ----------------------------------

    parent_ids = []

    for doc in retrieved_hq_docs:

        parent_ids.append(doc.metadata["parent_chunk_id"])

    # ----------------------------------
    # Step 3
    # Remove Duplicates
    # ----------------------------------

    unique_parent_ids = list(
        dict.fromkeys(parent_ids)
    )

    # ----------------------------------
    # Step 4
    # Retrieve Original Chunks
    # ----------------------------------

    original_chunks = (
        chunk_vectorstore.get(
            ids=unique_parent_ids,
            include=["documents"]
        )
    )

    # ----------------------------------
    # Step 5
    # Build Context
    # ----------------------------------

    context = "\n\n---\n\n".join(
        original_chunks["documents"]
    )

    # ----------------------------------
    # Step 6
    # Generate Final Answer
    # ----------------------------------

    answer = generate_answer(
        context=context,
        question=query
    )

    return {
        "query": query,
        "answer": answer,
        "retrieved_hq_docs": retrieved_hq_docs,
        "parent_chunk_ids": unique_parent_ids,
        "context": context
    }

# Test Complete Pipeline

In [209]:
result = answer_query(
    "What was Tesla automotive revenue in 2021?"
)
for key, value in result.items():
    print(f"{key}: {value}\n")


query: What was Tesla automotive revenue in 2021?

answer: $47,232 million.

retrieved_hq_docs: [Document(id='hq_text_30_0', metadata={'hq_id': 'hq_text_30_0', 'parent_chunk_id': 'text_30'}, page_content="What were Tesla's total revenues in 2021?"), Document(id='hq_text_2390_1', metadata={'hq_id': 'hq_text_2390_1', 'parent_chunk_id': 'text_2390'}, page_content="How much did Tesla's automotive sales revenue increase from 2020 to 2022?"), Document(id='hq_text_2390_0', metadata={'hq_id': 'hq_text_2390_0', 'parent_chunk_id': 'text_2390'}, page_content="What were Tesla's revenues in 2022?"), Document(id='hq_text_1061_0', metadata={'hq_id': 'hq_text_1061_0', 'parent_chunk_id': 'text_1061'}, page_content='What are the revenues of Tesla, Inc. for the year 2020?'), Document(id='hq_text_2393_2', metadata={'hq_id': 'hq_text_2393_2', 'parent_chunk_id': 'text_2393'}, page_content='What was the net income for Tesla in 2022?')]

parent_chunk_ids: ['text_30', 'text_2390', 'text_1061', 'text_2393']

co

In [210]:
original_chunks["metadatas"][0]

{'creationdate': '2022-05-02T10:10:26+00:00',
 'creator': 'wkhtmltopdf 0.12.5',
 'page': 9,
 'page_label': '10',
 'producer': 'Qt 5.11.3',
 'source': 'tesla-annual-reports\\tsla-10ka_20211231-gen.pdf',
 'title': '',
 'total_pages': 56}

# Benchmark Evaluation

In [219]:
# ==========================================
# Benchmark Questions
# ==========================================

benchmark_questions = {

    "HQ1": (
        "What should a board member ask about risks "
        "that could prevent Tesla from meeting production, "
        "delivery, or scaling expectations?"
    ),

    "HQ2": (
        "How should an analyst investigate the relationship "
        "between product defects, warranty or service obligations, "
        "customer trust, and brand risk?"
    ),

    "HQ3": (
        "What evidence helps determine whether future cash flow "
        "depends more on capital expenditure discipline, "
        "working capital, or operating income?"
    ),

    "HQ4": (
        "Which disclosures help evaluate technology, "
        "cybersecurity, data, or AI operational risk even "
        "if the user does not explicitly say cybersecurity?"
    )
}

# JSON Skeleton Creation 

In [220]:
import json

benchmark_results = []

for question_id, query in benchmark_questions.items():

    result = answer_query(query)

    record = {

        "question_id": question_id,

        "user_query": query,

        "retrieved_hypothetical_questions": [],

        "parent_chunks_used": [],

        "final_answer": result["answer"],

        "citations": [],

        "comparison_with_baseline": ""
    }

    # ----------------------------------
    # Retrieved HQ Questions
    # ----------------------------------

    for doc in result["retrieved_hq_docs"]:

        record[
            "retrieved_hypothetical_questions"
        ].append({

            "hypothetical_question":
                doc.page_content,

            "parent_chunk_id":
                doc.metadata.get(
                    "parent_chunk_id",
                    ""
                ),

            # Leave blank if metadata
            # not available
            "section":
                doc.metadata.get(
                    "section",
                    ""
                ),

            "year":
                doc.metadata.get(
                    "year",
                    ""
                ),

            "score":
                ""
        })

    # ----------------------------------
    # Parent Chunks Used
    # ----------------------------------

    for chunk_id in result[
        "parent_chunk_ids"
    ]:

        record[
            "parent_chunks_used"
        ].append({

            "chunk_id":
                chunk_id,

            "source_doc":
                "Tesla Annual Report",

            "section":
                "",

            "year":
                ""
        })

    benchmark_results.append(
        record
    )

print(
    f"Generated results for "
    f"{len(benchmark_results)} "
    f"benchmark questions."
)

Generated results for 4 benchmark questions.


In [221]:
# ==========================================
# Save Benchmark Results
# ==========================================

with open(
    "benchmark_results.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(benchmark_results, f, indent=4, ensure_ascii=False
    )

print(
    "benchmark_results.json saved successfully."
)

benchmark_results.json saved successfully.


In [222]:
benchmark_results[0]

{'question_id': 'HQ1',
 'user_query': 'What should a board member ask about risks that could prevent Tesla from meeting production, delivery, or scaling expectations?',
 'retrieved_hypothetical_questions': [{'hypothetical_question': "What are the potential risks and uncertainties that could affect Tesla's business?",
   'parent_chunk_id': 'text_911',
   'section': '',
   'year': '',
   'score': ''},
  {'hypothetical_question': "What are the potential risks that could harm Tesla's business, prospects, financial condition, and operating results?",
   'parent_chunk_id': 'text_918',
   'section': '',
   'year': '',
   'score': ''},
  {'hypothetical_question': "What are the potential risks that could harm Tesla's business, prospects, financial condition, and operating results?",
   'parent_chunk_id': 'text_919',
   'section': '',
   'year': '',
   'score': ''},
  {'hypothetical_question': "What are some of the risks that could harm Tesla's business, financial condition, and operating result